<a href="https://colab.research.google.com/github/rahiakela/genai-research-and-practice/blob/main/generative-ai-design-patterns/01_logits_masking/0_logits_masking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Masking out continuations that do not fit generation goals

We have a list of words that are forbidden, and another list of words that are brand words.
Choose continuations that do not use the forbidden words, and maximize the use of brand words.

In [ ]:
#%pip install --upgrade --quiet transformers torch fbgemm-gpu accelerate

In [ ]:
MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"

In [ ]:
!wget https://github.com/lakshmanok/generative-ai-design-patterns/raw/main/examples/01_logits_masking/banned_phrases.txt
!wget https://github.com/lakshmanok/generative-ai-design-patterns/raw/main/examples/01_logits_masking/desired_phrases.txt

## Word lists

Set up the word lists.

"Banned words" from https://channelkey.com/amazon-content-seo-and-optimization/400-restricted-amazon-keywords-the-most-comprehensive-list-youll-ever-need/

In [ ]:
# From https://channelkey.com/amazon-content-seo-and-optimization/400-restricted-amazon-keywords-the-most-comprehensive-list-youll-ever-need/
with open("banned_phrases.txt") as ifp:
    banned_phrases = [line.strip().lower() for line in ifp.readlines()]
banned_phrases[100:105]

['decomposable', 'definitive', 'degradable', 'dementia', 'depression']

In [ ]:
# Based on https://marketkeep.com/seo-keywords-for-nutrition/
with open("desired_phrases.txt") as ifp:
    desired_phrases = [line.strip().lower() for line in ifp.readlines()]
desired_phrases[:5]

['nutrition', 'calorie deficit', 'diet', 'protein shake', 'paleo diet']

In [ ]:
# makes unique
banned_phrases = set(banned_phrases)
desired_phrases = set(desired_phrases)

## Zero-shot generation

Without any logits processing

In [ ]:
from transformers import pipeline

pipe = pipeline(
    task="text-generation",
    model=MODEL_ID,
    use_fast=True,
    kwargs={
        "return_full_text": False,
    },
    model_kwargs={}
)

In [ ]:
def generate_product_description(item: str) -> str:
    system_prompt = f"""
        You are a product marketer for a company that makes nutrition supplements.
        Balance your product descriptions to attract customers, optimize SEO, and
        stay within accurate advertising guidelines.
        Product descriptions have to be 3-5 sentences.
        Provide only the product description with no preamble.
    """
    user_prompt = f"""
        Write a product description for a {item}.
    """

    input_message = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
    ]

    results = pipe(input_message,
                   max_new_tokens=512)
    return results[0]['generated_text'][-1]['content'].strip()

prod = generate_product_description("protein drink")
print(prod)

In [ ]:
import numpy as np

def evaluate(descr: str, positives, negatives) -> int:
    # go through and count the number of desired phrases and banned phrases
    descr = descr.lower()
    num_positive = np.sum([1 for phrase in positives if phrase in descr])
    num_negative = np.sum([1 for phrase in negatives if phrase in descr])
    return int(num_positive - num_negative)

def evaluate_verbose(descr: str, positives, negatives) -> int:
    # go through and count the number of desired phrases and banned phrases
    descr = descr.lower()

    num_positive = num_negative = 0
    for phrase in positives:
        if phrase in descr:
            num_positive += 1
            print(f"Good: {phrase}")
    for phrase in negatives:
        if phrase in descr:
            num_negative += 1
            print(f"Bad: {phrase}")
    print(f"Good: {num_positive}   Bad: {num_negative}")
    return num_positive - num_negative

evaluate_verbose(prod, desired_phrases, banned_phrases)

Good: whey
Good: whey protein
Bad: growth
Bad: perfect
Bad: quality
Good: 2   Bad: 3


-1

In [ ]:
evaluate(prod, desired_phrases, banned_phrases)

-1

## Use logits processing to enhance the product description

We'll use the same prompt, but use logits processing to upvote/downvote specific words

In [ ]:
import torch
import numpy as np
from transformers.generation.logits_process import (
    LogitsProcessor,
    LOGITS_PROCESSOR_INPUTS_DOCSTRING,
)
from transformers.utils import add_start_docstrings

class BrandLogitsProcessor(LogitsProcessor):
    def __init__(self, tokenizer, positives, negatives):
        self.tokenizer = tokenizer
        self.positives = positives
        self.negatives = negatives

    @add_start_docstrings(LOGITS_PROCESSOR_INPUTS_DOCSTRING)
    def __call__(
        self, input_ids: torch.LongTensor, input_logits: torch.FloatTensor
    ) -> torch.FloatTensor:
        output_logits = input_logits.clone()

        num_matches = [0] * len(input_ids)
        for idx, seq in enumerate(input_ids):
            # decode the sequence
            decoded = self.tokenizer.decode(seq)
            num_matches[idx] = evaluate(decoded, self.positives, self.negatives)
        max_matches = np.max(num_matches)

        # logits goes from -inf to zero.  Mask out the non-max sequences; torch doesn't like it to be -np.inf
        for idx in range(len(input_ids)):
            if num_matches[idx] != max_matches:
                output_logits[idx] = -10000

        return output_logits

In [ ]:
def generate_product_description_v2(item: str) -> str:
    system_prompt = f"""
        You are a product marketer for a company that makes nutrition supplements.
        Balance your product descriptions to attract customers, optimize SEO, and
        stay within accurate advertising guidelines.
        Product descriptions have to be 3-5 sentences.
        Provide only the product description with no preamble.
    """
    user_prompt = f"""
        Write a product description for a {item}.
    """

    input_message = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
    ]

    # alliterate on the first letter of the animal. So, donkey would be D
    brand_processor = BrandLogitsProcessor(pipe.tokenizer, desired_phrases, banned_phrases)

    results = pipe(input_message,
                   max_new_tokens=512,
                   do_sample=True,
                   temperature=0.8,
                   num_beams=10,
                   use_cache=True, # default is True
                   logits_processor=[brand_processor])
    return results[0]['generated_text'][-1]['content'].strip()

prod = generate_product_description_v2("protein drink")
print(prod)

In [ ]:
evaluate_verbose(prod, desired_phrases, banned_phrases)

Good: whey
Good: whey protein
Good: calorie
Good: 3   Bad: 0


3